# 22 · Exact differential operators as equivariant features

A recurring inductive bias for the physical world is **symmetry**. omnibias's
exact differential geometry gives equivariant feature extractors *for free*:

- **intrinsic** quantities (induced metric, scalar curvature) are **E(3)-invariant**;
- the **Jacobian** (a vector/tensor feature) is **equivariant**: `J → R J`.

We verify both to machine precision over many random rigid motions.

In [ ]:
import sys

import numpy as np
import torch
import torch.func as tf
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, PRIMARY, ACCENT, GOOD
set_style()

torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

from omnibias.geometry import ChartSpec, ManifoldSpec
from omnibias.geometry.torch import ops as geo

def rand_rotation():
    Q, _ = torch.linalg.qr(torch.randn(3, 3))
    if torch.det(Q) < 0:
        Q[:, 0] = -Q[:, 0]
    return Q

base = lambda x: torch.stack([torch.sin(x[0]) * torch.cos(x[1]),
                              torch.sin(x[0]) * torch.sin(x[1]),
                              torch.cos(x[0])])
coords = torch.tensor([[0.7, 0.3], [1.1, 1.5], [1.9, 2.2], [2.4, 4.0]])

## 1. Intrinsic features are E(3)-invariant

Apply a random rotation `R` and translation `t` to the embedding,
`φ_R = R·φ + t`. The pullback metric and the scalar curvature are unchanged.

In [ ]:
m_err, c_err = [], []
g0 = geo.pullback_metric(coords, ChartSpec(phi=base, domain_dim=2, ambient_dim=3))
man0 = ManifoldSpec("a", 2, geo.metric_spec_from_chart(ChartSpec(phi=base, domain_dim=2, ambient_dim=3)))
sc0 = geo.scalar_curvature(coords, man0)

for _ in range(60):
    R, t = rand_rotation(), torch.randn(3)
    phiR = (lambda x, R=R, t=t: R @ base(x) + t)
    cR = ChartSpec(phi=phiR, domain_dim=2, ambient_dim=3)
    m_err.append(float((geo.pullback_metric(coords, cR) - g0).abs().max()))
    manR = ManifoldSpec("b", 2, geo.metric_spec_from_chart(cR))
    c_err.append(float((geo.scalar_curvature(coords, manR) - sc0).abs().max()))

print(f"metric    invariance error: max = {max(m_err):.2e}")
print(f"curvature invariance error: max = {max(c_err):.2e}")

fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.hist(np.log10(np.array(m_err) + 1e-18), bins=20, color=PRIMARY, alpha=0.7, label="metric")
ax.hist(np.log10(np.array(c_err) + 1e-18), bins=20, color=ACCENT, alpha=0.7, label="curvature")
ax.set_xlabel("log₁₀ invariance error"); ax.set_title("E(3)-invariance (≈ machine precision)")
ax.legend()
plt.tight_layout()

## 2. The Jacobian is equivariant: `J → R J`

Tangent/vector features transform *with* the rotation. The exact Jacobian from
forward-mode autodiff satisfies `J(R·φ) = R · J(φ)` to machine precision.

In [ ]:
eq_err = []
J0 = tf.vmap(tf.jacfwd(base))(coords)
for _ in range(60):
    R = rand_rotation()
    phiR = (lambda x, R=R: R @ base(x))
    J1 = tf.vmap(tf.jacfwd(phiR))(coords)
    eq_err.append(float((J1 - torch.einsum("ij,bjk->bik", R, J0)).abs().max()))
print(f"Jacobian equivariance error: max = {max(eq_err):.2e}")

fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(eq_err, "o-", color=GOOD, ms=4)
ax.set_yscale("symlog", linthresh=1e-16)
ax.set_xlabel("random rotation #"); ax.set_ylabel("‖J(Rφ) − R·J(φ)‖∞")
ax.set_title("Vector-feature equivariance")
plt.tight_layout()

## 3. What omnibias already gives you — and what is deferred

- **Permutation equivariance / antisymmetry** for many-body systems (electrons,
  molecules) already lives in `omnibias-ferminet` (permutation-equivariant blocks
  + fermionic antisymmetry).
- **Diffeomorphism covariance** (general covariance) is built into
  `omnibias-geometry`'s exterior calculus and tensor operators.
- The exact operators here — `grad`, `div`, Laplacian, `laplace_beltrami`,
  curvature — are **equivariant feature extractors** you can drop into geometric
  or graph networks: scalar outputs are E(3)-invariant, vector/tensor outputs are
  equivariant.

**Deferred (different axis):** full **SE(3)/E(3) steerable** layers built from
SO(3) irreps, spherical harmonics, and Clebsch–Gordan tensor products (the
`e3nn` family). That is representation theory rather than differential calculus,
and would be a separate dependency/workstream rather than an omnibias primitive.